In [1]:
import glob
import pickle

pickle_file = "20241101_004535_ech_adaptation.pickle"


def load_data(filename_pattern):
    """
    Loads data from pickle files matching a pattern.

    Args:
        filename_pattern (str): A glob pattern to match pickle filenames.

    Returns:
        list: A list containing the data loaded from each pickle file.
    """

    pickle_files = glob.glob(filename_pattern)
    all_data = []

    for file in pickle_files:
        try:
            with open(file, "rb") as f:
                data = pickle.load(f)
                all_data.append(data)
        except Exception as e:
            print(f"Error loading data from {file}: {e}")

    return all_data


# Example usage:
filename_pattern = "./pickle/" + pickle_file
data = load_data(filename_pattern)

print(data)

[]


# ECH Adaptation Only


# ECH Distinct Configurations Only


# ECH Adaptation and Distinct Configurations Combined


In [4]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Data
data = {
    "Date": [
        "2024-08-08_18-18-05",
        "2024-08-19_19-04-54",
        "2024-08-21_13-59-28",
        "2024-09-05_01-33-25",
        "2024-09-06_01-04-14",
        "2024-09-08_14-44-57",
        "2024-09-11_10-14-23",
        "2024-09-15_21-43-55",
        "2024-10-13_02-50-30",
    ],
    "ECH Configurations": [
        30,
        15301,
        67979,
        297137,
        592346,
        592006,
        1029773,
        1177134,
        1165138,
    ],
    "Distinct ECH Configurations": [16, 25, 24, 26, 25, 25, 27, 24, 45],
}

# Create a DataFrame
df = pd.DataFrame(data)

# Convert the 'Date' column to datetime for better plotting
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d_%H-%M-%S")

# Add Cloudflare activation start and end dates
df = pd.concat(
    [
        df,
        pd.DataFrame(
            {
                "Date": pd.to_datetime(["2024-08-14", "2024-09-16"]),
                "ECH Configurations": [0, 0],
                "Distinct ECH Configurations": [0, 0],
            }
        ),
    ]
).sort_values(by="Date")

# Set Date as the index for easier plotting
df.set_index("Date", inplace=True)

# Set up the figure and axes
fig, ax = plt.subplots(figsize=(10, 8))

# Define bar width
width = 0.4

# --- Seaborn plotting ---
# Melt the DataFrame to long format for seaborn
df_melted = df.reset_index().melt(id_vars="Date", var_name="Configuration Type", value_name="Count")

# Convert 'Date' back to string for the y-axis
df_melted["Date"] = df_melted["Date"].dt.strftime("%Y-%m-%d")

# Plot with seaborn
sns.barplot(
    data=df_melted,
    x="Count",
    y="Date",
    hue="Configuration Type",
    orient="h",
    ax=ax,
    palette={"ECH Configurations": "#1f77b4", "Distinct ECH Configurations": "#2ca02c"},
    alpha=0.85,
    dodge=True,
)

# --- Reverse the y-axis ---
ax.invert_yaxis()

# --- Rest of the code (adjustments needed for annotations) ---

# Highlight Cloudflare activation period
activation_start = pd.to_datetime("2024-08-14").strftime("%Y-%m-%d")
activation_end = pd.to_datetime("2024-09-16").strftime("%Y-%m-%d")

# Determine maximum x-value for the plot (considering both data columns)
max_x = max(df["ECH Configurations"].max(), df["Distinct ECH Configurations"].max()) * 1.0

ax.barh(activation_start, max_x, height=width, color="orange", alpha=0.5)
ax.barh(activation_end, max_x, height=width, color="orange", alpha=0.5)

# --- Add labels with separate positioning ---
ax.annotate(
    "Cloudflare Activation Start",
    xy=(max_x, activation_start),
    xytext=(-5, 10),
    textcoords="offset points",
    color="orange",
    fontsize=12,
    ha="right",
    va="bottom",
)
ax.annotate(
    "Cloudflare Activation End",
    xy=(max_x, activation_end),
    xytext=(-5, -10),
    textcoords="offset points",
    color="orange",
    fontsize=12,
    ha="right",
    va="top",
)

# Set x-axis to log scale
ax.set_xscale("log")

# Hide axis titles and y-label
ax.set_xlabel("")
ax.set_ylabel("")

# ---  Use unique dates from df_melted for y-ticks ---
date_range = df_melted["Date"].unique()
ax.set_yticks(date_range)
ax.set_yticklabels(date_range, fontsize=12)

# Set tick parameters for x-axis
ax.tick_params(axis="x", which="major", labelsize=12)
ax.tick_params(axis="x", which="minor", labelsize=12)

# Drop ax Spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Add annotations for data values
# (This part needs adjustments since seaborn handles bar positions differently)
for i, (v1, v2) in enumerate(zip(df["ECH Configurations"], df["Distinct ECH Configurations"], strict=False)):
    if v1 > 0:
        ax.text(
            v1,
            i - 0.2,
            f"{v1}",
            ha="left",
            va="center",
            fontsize=10.5,
            fontweight="bold",
            color="#1f77b4",
        )
    if v2 > 0:
        ax.text(
            v2,
            i + 0.2,
            f"{v2}",
            ha="left",
            va="center",
            fontsize=10.5,
            fontweight="bold",
            color="#2ca02c",
        )

# Add legend
ax.legend(loc="lower right", fontsize=14)

# Tight layout for proper spacing
plt.tight_layout()

# Show plot
plt.show()

ModuleNotFoundError: No module named 'seaborn'